In [1]:
#setup installing package to the venv
%pip install pandas pyarrow pyspark
%pip install install-jdk

  Using cached pandas-3.0.5-cp314-cp314-macosx_11_0_arm64.whl.metadata (79 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 450.1/450.1 MB 3.8 MB/s  0:02:050:00:0100:04
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached numpy-2.5.2-cp314-cp314-macosx_14_0_arm64.whl.metadata (6.6 kB)
  Using cached py4j-0.10.9.9-py2.py3-none-any.whl.metadata (1.3 kB)
Using cached pandas-3.0.5-cp314-cp314-macosx_11_0_arm64.whl (10.0 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.9/35.9 MB 4.1 MB/s  0:00:08m0:00:0100:01
Using cached py4j-0.10.9.9-py2.py3-none-any.whl (203 kB)
Using cached numpy-2.5.2-cp314-cp314-macosx_14_0_arm64.whl (5.4 MB)
  Created wheel for pyspark: filename=pyspark-4.2.0-py2.py3-none-any.whl size=450798675 sha256=d57a7bbbcf18a2f0d15d45da09bf395effc962bb80c0ab5dd37b0ea5fc11e500
  Stored in directory: /Users/ozyohay/Library/Caches/pip/wheels/b2/35/e8/0f0feeae799066fbc77af369b7c2ea3f

In [10]:
#java home setup
#this is importent for the venv, not part of our project
%pip install install-jdk

import os
import sys
import glob
import jdk

# 1. Target the .venv directory of the current running Python kernel
venv_path = sys.prefix
jvm_dir = os.path.join(venv_path, "jvm")

# 2. Download/ensure JDK 17 exists for THIS current OS/architecture
jdk.install('17', path=jvm_dir)

# 3. Dynamically search for the 'bin/java' executable regardless of OS directory nesting
java_execs = glob.glob(os.path.join(jvm_dir, "**/bin/java"), recursive=True)
if not java_execs:
    # Check for Windows .exe just in case
    java_execs = glob.glob(os.path.join(jvm_dir, "**/bin/java.exe"), recursive=True)

if not java_execs:
    raise RuntimeError(f"JDK binary could not be found inside {jvm_dir}")

# The true JAVA_HOME is the parent directory of 'bin'
resolved_java_home = os.path.dirname(os.path.dirname(os.path.abspath(java_execs[0])))

# 4. Set environment variables for the active session
os.environ["JAVA_HOME"] = resolved_java_home
os.environ["PATH"] = os.path.join(resolved_java_home, "bin") + os.pathsep + os.environ.get("PATH", "")

print(f"Universal JDK configured at: {resolved_java_home}")

Note: you may need to restart the kernel to use updated packages.
✅ Universal JDK configured at: /Users/ozyohay/Documents/Code/repos/teleSpikeRedo/.venv/jvm/jdk-17.0.20+8/Contents/Home


In [11]:
import re

import sqlite3
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as SqlFun
from pyspark.sql.types import ArrayType, StringType


#def GetBaseline(token):

# a simple worker functaion for parsing one message into tokes
def tokenize(text):
    if not text:
        return []
    #Hebrew and alphanumeric words of length >= 2
    return re.findall(r"[\u0590-\u05fe\w]{2,}", text.lower())

#conver time stamp into hour and date
def add_time_columns(df):
    return df.withColumn("dt", SqlFun.to_timestamp(SqlFun.col("ts"))) \
             .withColumn("hour", SqlFun.hour(SqlFun.col("dt"))) \
             .withColumn("date", SqlFun.to_date(SqlFun.col("dt")))

#tokenise and exploead 
def explode_and_filter_tokens(df, allowed_tokens=None):
    tokenize_udf = SqlFun.udf(tokenize, ArrayType(StringType()))
    
    tokens_df = df.withColumn("word", SqlFun.explode(tokenize_udf(SqlFun.col("text"))))
    
    if allowed_tokens:
        tokens_df = tokens_df.filter(SqlFun.col("word").isin(allowed_tokens))
        
    return tokens_df

def compute_hourly_pivots(tokens_df, total_days):
    # group rows per word and hour
    counts = tokens_df.groupBy("word", "hour").count()
    
    # Pivot hours into columns (now each word has a clolumb for each time of day)
    pivoted = counts.groupBy("word").pivot("hour", list(range(24))).sum("count").na.fill(0)
    
    # divide counts by total days to get average baseline per hour
    # Rename columns to h0, h1 ... h23
    for h in range(24):
        pivoted = pivoted.withColumn(f"h{h}", SqlFun.col(str(h)) / total_days).drop(str(h))
        
    return pivoted

#main function calling other parts    
def generate_baseline_table(sqlite_path, output_table_path, allowed_tokens):
    #this will recive an sql full of thounsds or millions of messages and a 
    #list of importent tokens. it will clean and tokenise each message, filter not importent tokens like "and" "if". any thing that isnt in the list.
    #it will then save in a small sql table
    #a row for each token with a columb for each hour of the day, and save the avrage apperenses in that hour for each token. we can than devide by 60 or 240 to get baselines for our time window

    spark = SparkSession.builder \
        .appName("BaselineGenerator") \
        .config("spark.driver.memory", "2g") \
        .getOrCreate()

    # load messages from local data base into Spark
    conn = sqlite3.connect(sqlite_path)
    pdf = pd.read_sql_query("SELECT text, ts FROM messages", conn)
    conn.close()

    if pdf.empty:
        print("No messages found.")
        return

    raw_df = spark.createDataFrame(pdf)

    #add time of day columb to data frame, parsed from the timestamp that came with the message
    #as well as a date columb for later calcualtions
    timed_df = add_time_columns(raw_df)

    # Calculate total days for avrge calculations later on
    total_days = max(1, timed_df.select("date").distinct().count())

    # tokenization and explode into rows
    tokens_df = explode_and_filter_tokens(timed_df, allowed_tokens=allowed_tokens)

    # use the data to turn the table into each row has: word h0 h1...., in each cloumb have the avrage for that time of day
    baseline_matrix = compute_hourly_pivots(tokens_df, total_days)

    # save to sql
    baseline_pdf = baseline_matrix.toPandas()
    
    conn_out = sqlite3.connect(output_table_path)
    baseline_pdf.to_sql("token_baselines", conn_out, if_exists="replace", index=False)
    conn_out.close()
    
    print(f"Generated baselines for {len(baseline_pdf)} tokens over {total_days} days.")
    return baseline_pdf

In [8]:
#create baselines.db
conn = sqlite3.connect("baselines.db")
cursor = conn.cursor()

hour_cols = ", ".join([f"h{i} REAL" for i in range(24)])

cursor.execute(f"""
CREATE TABLE IF NOT EXISTS token_baselines (
    word TEXT,
    {hour_cols}
)
""")

conn.commit()
conn.close()

print("Empty token_baselines table created.")

Empty token_baselines table created.


In [12]:
# Run the pipeline on your scraped messages
baselines_df = generate_baseline_table(
    sqlite_path="messages.db",
    output_table_path="baselines.db",
    allowed_tokens=["טיל","ישראל"]  # Set to a list like ["טיל", "אזעקה"] if you want to filter, or None for all
)

# Preview the top tokens at 14:00 (2:00 PM)
if baselines_df is not None:
    print(baselines_df[["word", "h14"]].sort_values(by="h14", ascending=False).head(10))

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/21 13:54:11 WARN Utils: Your hostname, Ozs-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.100.102.19 instead (on interface en0)
26/08/21 13:54:11 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/Users/ozyohay/Documents/Code/repos/teleSpikeRedo/.venv/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/08/21 13:54:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
/Users/ozyohay/

Generated baselines for 2 tokens over 572 days.
    word       h14
0  ישראל  0.512238
1    טיל  0.010490


In [5]:
#print data base messages.db
conn = sqlite3.connect("messages.db")
df_msgs = pd.read_sql_query("SELECT * FROM messages LIMIT 20", conn)
conn.close()
print("Messages Table:")
display(df_msgs)

Messages Table:


,channel,sender,text,ts
0,קול החדשות - רגע NEWS🔴,-1001254976833,ניסיון לינץ' במטיילים באזור מעלה תידהר: עשרות ...,1787303338
1,קול החדשות - רגע NEWS🔴,-1001254976833,מטוס התדלוק השני מדגם Boeing KC-46 נחת אתמול ב...,1787298666
2,קול החדשות - רגע NEWS🔴,-1001254976833,"הסופרת המצרית, דליה זיאדה, שתמכה בישראל לאחר ט...",1787297698
3,קול החדשות - רגע NEWS🔴,-1001254976833,בשל השיבושים והחשש לאי הגעת מזוודות חברת אל על...,1787295408
4,קול החדשות - רגע NEWS🔴,-1001254976833,המחבל שחוסל הלילה על ידי לוחמי גדוד 890 של הצנ...,1787291624
5,קול החדשות - רגע NEWS🔴,-1001254976833,פרטים על התקרית הלילה בג'נין: לוחמי גדוד 890 ש...,1787287701
6,קול החדשות - רגע NEWS🔴,-1001254976833,תחזית מזג האוויר: היום צפויה עלייה נוספת בטמפר...,1787285036
7,קול החדשות - רגע NEWS🔴,-1001254976833,נסיון פיגוע בג'נין: במהלך פעילות התקפית של כוח...,1787284713
8,קול החדשות - רגע NEWS🔴,-1001254976833,מקור סורי מודה בשיחה עם העיתונאי ג'קי חוגי כי ...,1787256167
9,קול החדשות - רגע NEWS🔴,-1001254976833,רעידת אדמה בעוצמה 6.7 הורגשה בפרו שבדרום אמריקה,1787251806


In [6]:
#print basline.db
conn = sqlite3.connect("baselines.db")
df_base = pd.read_sql_query("SELECT word, h0, h8, h14, h20 FROM token_baselines ORDER BY h14 DESC LIMIT 10", conn)
conn.close()
print("Baselines Table:")
display(df_base)

DatabaseError: Execution failed on sql 'SELECT word, h0, h8, h14, h20 FROM token_baselines ORDER BY h14 DESC LIMIT 10': no such table: token_baselines